In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Lendo as tabelas do catálogo
df_users = spark.table("workspace.default.users_yt")
df_posts = spark.table("workspace.default.posts_creator")

# Criando Views Temporárias para usar SQL puro também
df_users.createOrReplaceTempView("users_yt")
df_posts.createOrReplaceTempView("posts_creator")

In [0]:
#Top 3 Posts por Likes (Últimos 6 meses)
six_months_ago = F.add_months(F.current_date(), -6)

# Nota: Se o número for muito grande (ex: 1700000000000), divida por 1000 (milissegundos)
df_posts_fixed = df_posts.withColumn(
    "published_at_ts", 
    F.from_unixtime(F.col("published_at") / 1000).cast("date") # Ajuste /1000 se for milissegundos
)

window_likes = Window.partitionBy("user_id").orderBy(F.col("likes").desc())

top_3_likes = (
    df_posts_fixed.join(df_users, df_posts_fixed.yt_user == df_users.user_id)
    .filter(F.col("published_at_ts") >= six_months_ago) # Agora comparamos DATE com DATE
    .withColumn("rank", F.row_number().over(window_likes))
    .filter(F.col("rank") <= 3)
    .select("user_id", "title", "likes", "rank")
)

top_3_likes.show()

In [0]:
#Top 3 Posts por Views (Últimos 6 meses)
query_top_views = """
SELECT * FROM (
    SELECT 
        u.user_id, 
        p.title, 
        p.views,
        ROW_NUMBER() OVER (PARTITION BY u.user_id ORDER BY p.views DESC) as rank
    FROM posts_creator p
    INNER JOIN users_yt u ON p.yt_user = u.user_id
    WHERE CAST(FROM_UNIXTIME(p.published_at / 1000) AS DATE) >= ADD_MONTHS(CURRENT_DATE(), -6)
) 
WHERE rank <= 3
"""

df_top_views = spark.sql(query_top_views)
display(df_top_views)

In [0]:
#Creators em 'posts_creator' ausentes em 'users_yt'

# Usando Left Anti Join no PySpark
mismatch_creators = df_posts.join(
    df_users, 
    df_posts.yt_user == df_users.user_id, 
    "left_anti"
).select(F.col("yt_user")).distinct()

print("Creators presentes em posts mas ausentes no cadastro de usuários:")
mismatch_creators.show()

In [0]:
#Análise Mensal

# 1. Preparar a base com Mês e Ano
df_monthly_base = df_posts.withColumn(
    "month", 
    F.date_format(F.from_unixtime(F.col("published_at") / 1000), "yyyy-MM")
)

df_pivot_analysis = (
    df_monthly_base
    .groupBy("yt_user")
    .pivot("month")
    .agg(F.count("title"))
    .fillna(0)  # Substitui os meses sem publicações (null) por 0
    .withColumnRenamed("yt_user", "user_id")
)

print("Quantidade de publicações por mês (Pivotado com tratamento de zeros):")
display(df_pivot_analysis)

In [0]:
###################################################Salvando os Resultados em Tabelas Delta################################################ (EXTRA)
# 1. Definindo o caminho e nome da tabela de performance mensal
df_top_views = spark.sql(query_top_views) 

# 2. Definindo o caminho e nome da tabela de performance mensal
target_analytics_table = "workspace.default.analytics_creators_monthly"

# Salvando o DataFrame Pivotado
(df_pivot_analysis.write
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(target_analytics_table))

# 3. Criando a tabela de Ranking (Unindo os dois DataFrames)
# Agora usamos df_top_views (DataFrame) em vez de query_top_views (String)
df_rankings = top_3_likes.unionByName(df_top_views, allowMissingColumns=True)

(df_rankings.write
    .mode("overwrite")
    .option("mergeSchema", "true") # Boa prática adicionar aqui também
    .saveAsTable("workspace.default.analytics_creators_ranking"))

print(f"✅ Tabelas de analytics criadas com sucesso no catálogo.")